# Systematic-review screening pipeline

Fill the `include` / `reason` columns of **`25 papers SP.xlsx`** reproducibly, and
**cross-check GPT-5.5 against a local model (qwen3-8b)** to catch the remaining
mistakes.

This notebook is deliberately thin: it *kicks off* the Python scripts in `pipeline/`
and *visualises / verifies* the output. All logic lives in the scripts, so it runs
the same from the terminal, local Jupyter, or Colab.

```
step0_load      xlsx            -> data/step0_papers.jsonl
step1_resolve   DOI             -> canonical URL + title/year checks   (catches wrong links)
step2_fetch     URL             -> raw text (HTML; PDF text-layer; GLM-OCR fallback)
step3_screen    text+metadata   -> include / reason           (run once per backend)
merge_results   all backends    -> 25 papers SP_screened.xlsx  (with disagreement flags)
```

**How the 3 mistakes surface:** Step 1 flags papers whose sheet link is wrong (e.g.
three papers sharing one ScienceDirect URL); the merge flags every paper where the
two models disagree, or where a model disagrees with the current human decision.
Sort the output sheet by `review_priority` and those rows sit at the top.

## Run environment

**Local (this machine):** select the `.venv` kernel, then just run the cells.

**Colab + VS Code + qwen3-8b on a GPU:**
1. Runtime -> Change runtime type -> **GPU** (T4 is enough in 4-bit; A100 for fp16).
2. Put this folder on Google Drive, mount it, and `cd` in (cell below auto-detects Colab).
3. qwen3-8b runs via HuggingFace `transformers` (default `QWEN_BACKEND=hf`) - no server needed.
   To instead use Ollama/LM Studio locally, set `QWEN_BACKEND=openai` (see the qwen cell).

In [ ]:
# --- environment setup (local or Colab) ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # EDIT this to where you put the folder on your Drive:
    PROJECT = Path("/content/drive/MyDrive/Noa vd Ploeg/Review")
    os.chdir(PROJECT)
else:
    PROJECT = Path.cwd()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Project:", PROJECT)
print("Colab:", IN_COLAB)

In [ ]:
# --- API keys + backend choice ---
import os, getpass
from pipeline import config, report   # importing config loads .env

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Which backends to run and compare. Drop 'qwen3-8b' if you have no GPU/local server yet.
BACKENDS = ["gpt-5.5", "qwen3-8b"]
print("Backends registered:", list(config.BACKENDS))

## Step 0 - load the workbook into a clean JSONL

In [ ]:
!python -m pipeline.step0_load

In [ ]:
report.papers_df()

## Step 1 - resolve DOIs to the correct URL

Look at the `flags` column. `duplicate_link_shared_with:...` means several papers
carry the *same* link (a data-entry error); `title_mismatch` means the DOI does not
match the title in the sheet. These are the rows whose old decisions are least
trustworthy, because the model may have read the wrong article.

In [ ]:
!python -m pipeline.step1_resolve

In [ ]:
# Only the papers with a data-quality problem:
report.flagged_df().style.set_properties(**{'background-color': '#fff3cd'}, subset=['flags'])

## Step 2 - fetch the raw article text

HTML pages are scraped; PDFs use the embedded text layer. OCR runs **only as a last
resort** - a PDF is OCR'd only when it has essentially no text layer (a genuine scan).
OCR is **free and keyless** via local [`glm-ocr` on Ollama](https://ollama.com/library/glm-ocr):

```bash
ollama pull glm-ocr   # one time; then `ollama serve` (usually already running)
```

If Ollama isn't running, Step 2 logs `OCR skipped` and keeps the abstract - it never
stalls. Publisher pages that block scraping show up as `http_error`; that is fine, the
sheet abstract is still used for screening. A live progress bar shows ETA; re-run with
`--only-missing` to fill just the gaps, or `--no-ocr` to disable OCR.

In [ ]:
!python -m pipeline.step2_fetch_text --sleep 1

In [ ]:
report.text_df()

## Step 3a - screen with GPT-5.5 (OpenAI)

In [ ]:
!python -m pipeline.step3_screen --backend gpt-5.5 --only-missing

In [ ]:
report.screen_df('gpt-5.5')[
    ['record_id','decision','include','reason','needs_human_review']]

## Step 3b - screen with qwen3-8b (local / Colab GPU)

Default is HuggingFace `transformers` (`QWEN_BACKEND=hf`), which is ideal on a Colab
GPU. On a T4, add 4-bit: `os.environ['QWEN_4BIT']='1'` before running.

Prefer a local **Ollama / LM Studio** server instead? Set these *before* the run:
```python
os.environ['QWEN_BACKEND']='openai'
os.environ['QWEN_BASE_URL']='http://localhost:11434/v1'   # Ollama
os.environ['QWEN_MODEL']='qwen3:8b'
```
The backend registry re-reads env at import, so restart the kernel after changing it,
or edit `pipeline/config.py`.

In [ ]:
# Uncomment to run. Needs a GPU (HF) or a running local server (openai mode).
# !python -m pipeline.step3_screen --backend qwen3-8b --only-missing

In [ ]:
# report.screen_df('qwen3-8b')[
#     ['record_id','decision','include','reason','needs_human_review']]

## Step 4 - merge, compare, and verify

`merge_results` writes **`25 papers SP_screened.xlsx`**, sorted so the rows most
likely to contain mistakes are at the top:

| priority | meaning |
|---|---|
| `DISAGREE-MODELS` | GPT-5.5 and qwen3-8b give different include decisions |
| `DISAGREE-HUMAN` | the models agree, but disagree with the current sheet |
| `FLAGGED-STEP1` | wrong/duplicate link or DOI-title mismatch |
| `NEEDS-REVIEW` | a model asked for human review |

In [ ]:
!python -m pipeline.merge_results --backends gpt-5.5 qwen3-8b

In [ ]:
report.summary(BACKENDS)

In [ ]:
report.decision_counts(BACKENDS)

### The shortlist to eyeball

These are your candidate mistakes. Read `*_reason` next to `human_reason` and decide.
Once you agree with a model, copy its `include`/`reason` into the master sheet.

In [ ]:
cmp = report.disagreements(BACKENDS)
cols = ['record_id','short_title','review_priority','human_include'] + \
       [c for c in cmp.columns if c.endswith('_include')] + \
       ['step1_flags']
cmp[cols]

In [ ]:
# Full reasoning for the shortlist (wraps text):
import pandas as pd
pd.set_option('display.max_colwidth', 90)
reason_cols = ['record_id','human_reason'] + [c for c in cmp.columns if c.endswith('_reason')]
cmp[reason_cols]